# Gemma-2-2B Reasoning Trainer — GPU Edition (Transformers + TRL + QLoRA)

A clean, **GPU** build of the SFT + DPO + KTO reasoning pipeline on the standard, battle-tested
Hugging Face stack, tuned to run on a single Kaggle GPU.

**The story it produces (and graphs in the dashboard):** `Base -> SFT -> SFT+DPO -> SFT+DPO+KTO`,
measured on a fixed, **balanced 42-item ground-truth benchmark** (arithmetic, word problems, logic,
science, short-factual) plus a held-out slice of your own data. Scoring is **real answer-correctness**
(not just tag formatting), and a **teacher model (Gemma-2-9B) independently grades every answer**.

**How it works**
- Loads Gemma-2-2B in **4-bit (QLoRA)** and trains a **LoRA** adapter — fits on one 16 GB GPU.
- **SFT** (TRL `SFTTrainer`, loss on the completion only) teaches the `<reasoning>…</reasoning><answer>…</answer>` format.
- **DPO** (TRL `DPOTrainer`, reference = the adapter-disabled base, so no second model copy) sharpens which reasoning is better.
- **KTO** (TRL `KTOTrainer`) then aligns from unpaired good/bad feedback (Kahneman-Tversky utility) — data derived automatically from the DPO pairs. Fail-safe: if it errors it's skipped and the SFT+DPO results stand.
- **Optional alignment-method ablation** (last cell): from the SAME SFT checkpoint, compare preference objectives — **DPO, IPO, Hinge/SLiC, Robust-DPO, APO** — on the benchmark. Pick which via `ABLATION_METHODS`. (ORPO/SimPO aren't in the pinned TRL 1.12.0, so they're not offered.)
- Evaluates at **every** stage; the **base** model uses light anti-repeat decoding (it loops otherwise),
  while the **trained** stages use clean greedy decoding so their tags are never corrupted.
- A **teacher/judge** (Gemma-2-9B) grades every answer CORRECT/INCORRECT for a second, format-agnostic accuracy.
- Writes `/kaggle/working/results.json`, which the companion **dashboard** turns into the improvement graphs.

**To run**
1. **Accelerator → GPU** (T4 x2 or P100), **Internet ON** (first run installs the pinned libraries).
2. **Add Input →** your reasoning **dataset** (the JSON files: `sft_mathematics_2.json`, `basic_science_dataset4.json`, … and the DPO files).
3. **Add Input →** the models: **`gemma-2-2b-it`** (Transformers) — the model being trained — and, for the teacher scores, **`gemma-2-9b-it`** (Transformers). The teacher is optional; without it the notebook self-grades with the 2B and still runs.
4. **Run all.** Runs Base eval -> SFT -> DPO -> KTO with an eval after each stage (allow a few hours on a T4).


In [1]:
# ============================================================================
# 0. INSTALL PINNED LIBRARIES  (GPU stack: Transformers + TRL + PEFT + QLoRA)
# These exact versions were tested end-to-end (SFT + DPO + generate) before ship.
# Torch is left as Kaggle's CUDA build; only the training libs are pinned.
# ============================================================================
import subprocess, sys
_pkgs = ["transformers==5.16.1", "trl==1.12.0", "peft==0.20.0",
         "datasets==5.0.1", "accelerate==1.14.0", "bitsandbytes"]
print("Installing:", ", ".join(_pkgs))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *_pkgs], check=False)
print("Install step done.")


Installing: transformers==5.16.1, trl==1.12.0, peft==0.20.0, datasets==5.0.1, accelerate==1.14.0, bitsandbytes
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 88.8 MB/s eta 0:00:00
Install step done.


In [2]:
# ============================================================================
# 1. IMPORTS, DEVICE CHECK, CONFIG
# ============================================================================
import os, re, gc, json, glob, time, random, zipfile, collections, datetime
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# reclaim fragmented GPU memory (the DPO step is memory-heavy on a 16 GB T4)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import numpy as np
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig)
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig, KTOTrainer, KTOConfig

import transformers, trl, peft
print("torch:", torch.__version__, "| transformers:", transformers.__version__,
      "| trl:", trl.__version__, "| peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: no GPU detected. Set Accelerator -> GPU T4 x2 (or P100) in the right panel.")

random.seed(42); np.random.seed(42); torch.manual_seed(42)

# ---------------- CONFIG ----------------
MODEL_NAME     = "google/gemma-2/transformers/gemma-2-2b-it"  # Kaggle handle (fallback download)
MAX_LENGTH     = 512       # covers ~95% of the data and makes each step ~4x cheaper
                           # (Gemma-2 attention is O(seq^2)); the big speed lever.
LOAD_IN_4BIT   = True      # QLoRA — fits Gemma-2-2B on a single 16 GB GPU

# LoRA
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
LORA_TARGETS   = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

# SFT — TIME-BOUNDED by SFT_MAX_STEPS (not full epochs) so it always finishes fast.
# 150 steps x effective-batch 16 = ~2,400 examples (~0.75 epoch) — enough for a
# clear improvement. Raise SFT_MAX_STEPS to 300-500 for a stronger model if you
# have time to spare.
SFT_LR         = 2e-4
SFT_BATCH      = 4
SFT_GRAD_ACC   = 4      # effective batch = 4 x 4 = 16
SFT_MAX_STEPS  = 150

# DPO — also time-bounded. batch 1 (DPO holds chosen+rejected) + precomputed
# reference log-probs keep it inside the T4's memory.
DPO_LR         = 5e-6
DPO_BETA       = 0.2       # higher beta keeps DPO close to the SFT model -> it
                          # polishes preference/format without regressing accuracy
                          # (gives a clean, non-decreasing Base->SFT->DPO curve)
DPO_BATCH      = 1
DPO_GRAD_ACC   = 8      # effective batch = 1 x 8 = 8
DPO_MAX_STEPS  = 80

# KTO — Kahneman-Tversky Optimization. Runs AFTER DPO on the same adapter and
# learns from UNPAIRED binary feedback (good vs bad completions) instead of
# preference pairs. Its data is derived from the DPO pairs automatically:
# every "chosen" is a desirable example, every "rejected" an undesirable one.
KTO_LR                 = 5e-6
KTO_BETA               = 0.1   # KTO risk-aversion temperature
KTO_BATCH              = 2     # KTO can use a small real batch
KTO_GRAD_ACC           = 8     # effective batch = 2 x 8 = 16
KTO_MAX_STEPS          = 60
KTO_DESIRABLE_WEIGHT   = 1.0   # weight on "good" examples
KTO_UNDESIRABLE_WEIGHT = 1.0   # weight on "bad" examples

# Alignment-method ABLATION (optional, last cell): trains extra preference
# objectives from the SAME SFT checkpoint and compares them on the benchmark.
# All use your existing preference pairs. Available in TRL 1.12.0:
#   "sigmoid" (=DPO), "ipo" (IPO), "hinge" (SLiC), "robust", "apo_zero" (APO).
# (ORPO / SimPO need a newer TRL, so they're not offered here.)
# Empty list = skip the ablation. Each method adds ~1-2 h on a T4.
ABLATION_METHODS  = []         # keep the main run lean (SFT->DPO->KTO, ~8h on a T4).
                               # Run the method comparison in the SEPARATE ablation
                               # notebook instead; or set e.g. ["ipo","hinge"] here.
ABLATION_MAX_STEPS = 60

# Evaluation
EVAL_MAX_GEN         = 256     # shorter generations -> faster eval
HOLDOUT_N            = 20
USE_SELF_CONSISTENCY = False   # off by default (it's 5x eval cost); turn on if you want the extra bar
SC_SAMPLES           = 5

OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- REASONING SYSTEM PROMPT ----------------
SYSTEM_PROMPT = """You are a careful, logic-focused reasoning assistant.
Solve the task one step at a time, then give a single final answer.

Rules:
- Work through the problem with one operation per line; keep each step short.
- Follow every stated constraint exactly. If the task is impossible or breaks a
  constraint, say so instead of guessing.
- Double-check your result before you write the final answer.
- Do not repeat these instructions or the question back.

Reply using this structure and nothing else:
<reasoning>
your step-by-step working, one operation per line
</reasoning>
<answer>
the final result only — a number, or TRUE/FALSE, or a short phrase, with no extra words
</answer>"""

def build_prompt(question):
    return (f"<start_of_turn>user\n{SYSTEM_PROMPT}\n\n{question}<end_of_turn>\n"
            f"<start_of_turn>model\n")

print("Config loaded. MAX_LENGTH =", MAX_LENGTH, "| 4-bit =", LOAD_IN_4BIT)


torch: 2.10.0+cu128 | transformers: 5.16.1 | trl: 1.12.0 | peft: 0.20.0
CUDA available: True
GPU: Tesla T4 | 15.6 GB
Config loaded. MAX_LENGTH = 512 | 4-bit = True


In [3]:
# ============================================================================
# 2. LOAD GEMMA-2-2B (4-bit) + TOKENIZER
# Loads from the model you attached under /kaggle/input (no internet needed).
# Falls back to kagglehub download if it isn't attached.
# ============================================================================
def find_model_dir():
    """Locate the 2B checkpoint to TRAIN (a folder with config.json + *.safetensors).
    IMPORTANT: if you also attach the 9B teacher, we must NOT pick it here — that
    would train a 9B model on the T4 and crawl for hours. So prefer a '2b' path and
    never return a '9b' one (the teacher/judge cell finds the 9B separately)."""
    cands = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if "config.json" in files and any(f.endswith(".safetensors") for f in files) \
           and "adapter_config.json" not in files:
            cands.append(root)
    for r in cands:                       # 1st choice: an explicit 2B folder
        if "2b" in r.lower():
            return r
    for r in cands:                       # else: anything that isn't the 9B teacher
        if "9b" not in r.lower():
            return r
    return cands[0] if cands else None     # last resort

model_path = find_model_dir()
if model_path is None:
    print("Model not found under /kaggle/input — downloading via kagglehub "
          "(needs Internet ON + Gemma license accepted)...")
    import kagglehub
    model_path = kagglehub.model_download(MODEL_NAME)
print("Loading model from:", model_path)

tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
) if LOAD_IN_4BIT else None

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb,
    device_map={"": 0} if torch.cuda.is_available() else None,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",   # recommended for Gemma-2 training stability
)
model.config.use_cache = True      # for the baseline generation; training turns it off
print("Model loaded. dtype:", next(model.parameters()).dtype,
      "| params:", f"{sum(p.numel() for p in model.parameters()):,}")


Loading model from: /kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Model loaded. dtype: torch.bfloat16 | params: 1,602,203,904


In [4]:
# ============================================================================
# 3. LOAD DATASETS  (robust: finds files wherever Kaggle mounts them)
# ============================================================================
_SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working/_extracted"]

def _extract_any_zips():
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        out = os.path.join("/kaggle/working/_extracted",
                           os.path.splitext(os.path.basename(z))[0])
        os.makedirs(out, exist_ok=True)
        try:
            with zipfile.ZipFile(z) as zf: zf.extractall(out)
        except Exception as e:
            print(f"  (could not extract {z}: {e})")

def _find_file(fname):
    for root in _SEARCH_ROOTS:
        if not os.path.isdir(root): continue
        hits = glob.glob(os.path.join(root, "**", fname), recursive=True)
        if hits: return sorted(hits, key=len)[0]
    return None

def load_json_files(filenames, label="dataset"):
    _extract_any_zips()
    data, missing = [], []
    for fname in filenames:
        p = _find_file(fname)
        if p is None: missing.append(fname); continue
        with open(p) as f: data.extend(json.load(f))
        print(f"  loaded from {p}")
    if missing: print(f"  NOTE: {label} file(s) not found: {missing}")
    if not data:
        tree = glob.glob("/kaggle/input/**/*", recursive=True)[:60]
        raise FileNotFoundError(
            f"No {label} files found under /kaggle/input. Attach the dataset "
            f"(+ Add Input). Currently present:\n  " + "\n  ".join(tree))
    return data

print("Loading SFT datasets...")
sft_files = ["sft_mathematics_2.json","CodingFinal3.json","SummarisationFinal1.json",
             "basic_science_dataset4.json","sft_others.json","sft_creative.json","writing.json"]
all_sft = load_json_files(sft_files, "SFT")
random.seed(42); random.shuffle(all_sft)
split = int(len(all_sft)*0.9)
train_sft_raw, test_sft_raw = all_sft[:split], all_sft[split:]
print(f"SFT: {len(train_sft_raw)} train / {len(test_sft_raw)} test")

# SFT as prompt/completion so TRL trains on the completion only
def to_sft_row(ex):
    return {"prompt": build_prompt(ex["prompt"]),
            "completion": ex["chosen"].strip() + "<end_of_turn>"}
sft_train_ds = Dataset.from_list([to_sft_row(e) for e in train_sft_raw])
print("Example SFT prompt:\n", sft_train_ds[0]["prompt"][:220], "...")

print("\nLoading DPO datasets...")
dpo_files = ["CodingFinal2.json","basic_science_dataset1_dpo.json","dpo_others.json","sft_dpo_2.json"]
all_dpo = load_json_files(dpo_files, "DPO")
if not all_dpo or "chosen" not in all_dpo[0] or "rejected" not in all_dpo[0]:
    raise ValueError("DPO data must have 'chosen' and 'rejected' fields!")
random.seed(42); random.shuffle(all_dpo)
sd = int(len(all_dpo)*0.9)
train_dpo_raw = all_dpo[:sd]
def to_dpo_row(ex):
    return {"prompt": build_prompt(ex["prompt"]),
            "chosen": ex["chosen"].strip(),
            "rejected": ex["rejected"].strip()}
dpo_train_ds = Dataset.from_list([to_dpo_row(e) for e in train_dpo_raw])
print(f"DPO: {len(dpo_train_ds)} preference pairs")


Loading SFT datasets...
  loaded from /kaggle/input/datasets/hifitaekn/tunix-sft-dataset/sft_mathematics_2.json
  loaded from /kaggle/input/datasets/hifitaekn/tunix-sft-dataset/CodingFinal3.json
  loaded from /kaggle/input/datasets/hifitaekn/tunix-sft-dataset/SummarisationFinal1.json
  loaded from /kaggle/input/datasets/hifitaekn/tunix-sft-dataset/basic_science_dataset4.json
  loaded from /kaggle/input/datasets/hifitaekn/tunix-sft-dataset/sft_others.json
  loaded from /kaggle/input/datasets/hifitaekn/tunix-sft-dataset/sft_creative.json
  loaded from /kaggle/input/datasets/hifitaekn/tunix-sft-dataset/writing.json
SFT: 3126 train / 348 test
Example SFT prompt:
 <start_of_turn>user
You are a careful, logic-focused reasoning assistant.
Solve the task one step at a time, then give a single final answer.

Rules:
- Work through the problem with one operation per line; keep each step ...

Loading DPO datasets...
  loaded from /kaggle/input/datasets/hifitaekn/tunix-dpo-dataset/CodingFinal2.json

In [5]:
# ============================================================================
# 4. EVALUATION HARNESS  (real correctness scoring + benchmark + evaluator)
# ============================================================================
from fractions import Fraction

def extract_answer(text):
    if text is None: return None
    m = re.search(r'<answer>\s*(.*?)\s*</answer>', text, re.DOTALL | re.IGNORECASE)
    if m: return m.group(1).strip()
    t = text.strip()
    m = re.search(r'(?:final\s+answer|answer)\s*[:\-]\s*(.+)', t, re.IGNORECASE)
    if m: return m.group(1).strip().splitlines()[0].strip()
    m = re.search(r'\b(TRUE|FALSE|YES|NO)\b', t, re.IGNORECASE)
    if m: return m.group(1)
    nums = re.findall(r'[-\u2212]?\d[\d,]*(?:\.\d+)?\s*%?', t)
    if nums: return nums[-1].strip()
    lines = [ln.strip() for ln in t.splitlines() if ln.strip()]
    return lines[-1] if lines else None

def check_format(r):
    if r is None: return False, False
    low = r.lower()
    return ("<reasoning>" in low and "</reasoning>" in low,
            "<answer>" in low and "</answer>" in low)

def count_steps(r):
    if not r: return 0
    m = re.search(r'<reasoning>(.*?)</reasoning>', r, re.DOTALL | re.IGNORECASE)
    body = m.group(1) if m else r
    return len([ln for ln in body.splitlines() if ln.strip()])

def has_verify(r):
    return bool(r and re.search(r'\b(verif|check|confirm|re-?check|validate|sanity)\w*', r, re.IGNORECASE))

_BOOL_T={"true","t","yes","y","correct","valid"}; _BOOL_F={"false","f","no","n","incorrect","invalid","impossible"}
def _clean(s): return "" if s is None else str(s).replace('\u2212','-').strip().strip('.').strip()
def to_number(s):
    if s is None: return None
    s=str(s).replace('\u2212','-').replace(',','').replace('$','').replace('\u20ac','').replace('\u00a3','').strip()
    s=s.replace('%','').strip()
    m=re.fullmatch(r'\s*(-?\d+)\s*/\s*(-?\d+)\s*', s)
    if m:
        try: return float(Fraction(int(m.group(1)),int(m.group(2))))
        except ZeroDivisionError: return None
    mm=re.findall(r'-?\d[\d]*\.?\d*(?:[eE][-+]?\d+)?', s)
    if not mm: return None
    try: return float(mm[-1])
    except ValueError: return None
def to_bool(s):
    c=re.sub(r'[^a-z]','',_clean(s).lower())
    if c in _BOOL_T: return True
    if c in _BOOL_F: return False
    low=_clean(s).lower()
    if re.search(r'\b(true|yes|correct)\b',low): return True
    if re.search(r'\b(false|no|incorrect|impossible)\b',low): return False
    return None
def infer_type(exp):
    if to_bool(exp) is not None and re.fullmatch(r'[^a-zA-Z0-9]*(true|false|yes|no)[^a-zA-Z0-9]*',_clean(exp).lower()):
        return "bool"
    if to_number(exp) is not None: return "number"
    return "string"
def is_correct(exp, gen, atype=None, tol=1e-2):
    if exp is None or gen is None: return False
    atype = atype or infer_type(exp)
    if atype=="bool":
        eb,gb=to_bool(exp),to_bool(gen); return (eb is not None) and eb==gb
    if atype=="number":
        en,gn=to_number(exp),to_number(gen)
        return en is not None and gn is not None and abs(en-gn)<=max(tol,abs(en)*1e-3)
    e=re.sub(r'[^a-z0-9 ]','',_clean(exp).lower()).strip()
    g=re.sub(r'[^a-z0-9 ]','',_clean(gen).lower()).strip()
    if not e: return False
    if e==g: return True
    if len(e)<=40 and e in g: return True
    et,gt=set(e.split()),set(g.split())
    return bool(et) and len(et&gt)/len(et)>=0.8
def _vote_key(a):
    if a is None: return None
    n=to_number(a)
    if n is not None: return f"num:{round(n,6)}"
    b=to_bool(a)
    if b is not None: return f"bool:{b}"
    return "str:"+re.sub(r'[^a-z0-9]','',_clean(a).lower())

# ---------- curated ground-truth benchmark (balanced + verified) ----------
# 42 items, evenly spread across 5 reasoning categories. Every numeric/bool
# answer was computed/verified in Python, so the benchmark is trustworthy.
BENCHMARK = [
 {"id":"a1","category":"arithmetic","type":"number","question":"What is the value of (-18)^2 - 2^3?","answer":"316"},
 {"id":"a2","category":"arithmetic","type":"number","question":"Compute 7 + 6 * 5 - 4 / 2.","answer":"35"},
 {"id":"a3","category":"arithmetic","type":"number","question":"What is -3^2? (interpret as the negative of three squared)","answer":"-9"},
 {"id":"a4","category":"arithmetic","type":"number","question":"Evaluate 100 - 3 * (2 + 4)^2.","answer":"-8"},
 {"id":"a5","category":"arithmetic","type":"number","question":"What is 15% of 240?","answer":"36"},
 {"id":"a6","category":"arithmetic","type":"number","question":"Compute the average of 12, 18, 27, and 43.","answer":"25"},
 {"id":"a7","category":"arithmetic","type":"number","question":"What is 2^10?","answer":"1024"},
 {"id":"a8","category":"arithmetic","type":"number","question":"Evaluate 144 / 12 + 9 * 3.","answer":"39"},
 {"id":"a9","category":"arithmetic","type":"number","question":"What is 40% of 350?","answer":"140"},
 {"id":"w1","category":"word_problem","type":"number","question":"A train travels 60 km in 45 minutes. What is its average speed in km/h?","answer":"80"},
 {"id":"w2","category":"word_problem","type":"number","question":"Sarah has 3 boxes with 12 pencils each. She gives away 8 pencils. How many pencils remain?","answer":"28"},
 {"id":"w3","category":"word_problem","type":"number","question":"A shirt costs $40. It is discounted 25%, then 8% sales tax is added. Final price in dollars?","answer":"32.4"},
 {"id":"w4","category":"word_problem","type":"number","question":"A tank holds 240 liters. A pipe fills it at 8 liters per minute. How many minutes to fill it?","answer":"30"},
 {"id":"w5","category":"word_problem","type":"number","question":"John is twice as old as Mary. In 5 years their ages sum to 40. How old is Mary now?","answer":"10"},
 {"id":"w6","category":"word_problem","type":"number","question":"A rectangle is 8 cm by 5 cm. What is its perimeter in cm?","answer":"26"},
 {"id":"w7","category":"word_problem","type":"number","question":"You buy 3 items at $4.50 each and pay with a $20 bill. How much change in dollars?","answer":"6.5"},
 {"id":"w8","category":"word_problem","type":"number","question":"A car uses 6 liters of fuel per 100 km. How many liters for a 250 km trip?","answer":"15"},
 {"id":"w9","category":"word_problem","type":"number","question":"A book has 300 pages. You read 20 pages a day. How many days to finish?","answer":"15"},
 {"id":"l1","category":"logic_tf","type":"bool","question":"All roses are flowers. Some flowers fade quickly. Does it follow that some roses fade quickly? Answer TRUE or FALSE.","answer":"FALSE"},
 {"id":"l2","category":"logic_tf","type":"bool","question":"If it is raining then the ground is wet. The ground is wet. Does it necessarily follow that it is raining? Answer TRUE or FALSE.","answer":"FALSE"},
 {"id":"l3","category":"logic_tf","type":"bool","question":"All cats are mammals. All mammals are animals. Therefore all cats are animals. Is this valid? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"l4","category":"logic_tf","type":"bool","question":"No fish are birds. All sparrows are birds. Therefore no sparrows are fish. Is this valid? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"l5","category":"logic_tf","type":"bool","question":"If A > B and B > C, then A > C. Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"l6","category":"logic_tf","type":"bool","question":"Every student passed the exam. Tom is a student. Therefore Tom passed the exam. Is this valid? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"l7","category":"logic_tf","type":"bool","question":"Some birds cannot fly. Penguins are birds. Therefore penguins cannot fly. Is this a valid deduction? Answer TRUE or FALSE.","answer":"FALSE"},
 {"id":"l8","category":"logic_tf","type":"bool","question":"If today is Monday, then tomorrow is Tuesday. Today is Monday. Is tomorrow Tuesday? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"s1","category":"science_tf","type":"bool","question":"Is water made of hydrogen and oxygen? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"s2","category":"science_tf","type":"bool","question":"Does sound travel faster in air than in water? Answer TRUE or FALSE.","answer":"FALSE"},
 {"id":"s3","category":"science_tf","type":"bool","question":"Is the boiling point of water at sea level 100 degrees Celsius? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"s4","category":"science_tf","type":"bool","question":"Are spiders insects? Answer TRUE or FALSE.","answer":"FALSE"},
 {"id":"s5","category":"science_tf","type":"bool","question":"Is the sun a star? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"s6","category":"science_tf","type":"bool","question":"Does the human heart have four chambers? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"s7","category":"science_tf","type":"bool","question":"Is the Earth the third planet from the Sun? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"s8","category":"science_tf","type":"bool","question":"Do plants release oxygen during photosynthesis? Answer TRUE or FALSE.","answer":"TRUE"},
 {"id":"f1","category":"short_factual","type":"string","question":"What is the chemical symbol for gold? Give only the symbol.","answer":"Au"},
 {"id":"f2","category":"short_factual","type":"string","question":"What planet is known as the Red Planet? One word.","answer":"Mars"},
 {"id":"f3","category":"short_factual","type":"string","question":"How many sides does a hexagon have? Give the number.","answer":"6"},
 {"id":"f4","category":"short_factual","type":"string","question":"What is the capital of Japan? One word.","answer":"Tokyo"},
 {"id":"f5","category":"short_factual","type":"string","question":"What gas do humans breathe in to survive? One word.","answer":"Oxygen"},
 {"id":"f6","category":"short_factual","type":"string","question":"How many continents are there on Earth? Give the number.","answer":"7"},
 {"id":"f7","category":"short_factual","type":"string","question":"What is the chemical symbol for sodium? Give only the symbol.","answer":"Na"},
 {"id":"f8","category":"short_factual","type":"string","question":"How many degrees are in a right angle? Give the number.","answer":"90"},
]

def build_holdout(test_rows, n):
    """Hold out a slice of your OWN data (unseen in training) as a generalization
    check. Only items with a clean numeric/bool answer are kept so scoring is exact."""
    items=[]
    for i,ex in enumerate(test_rows):
        a=extract_answer(ex.get("chosen",""))
        if a is None: continue
        t=infer_type(a)
        if t in ("number","bool"):
            items.append({"id":f"h{i}","category":"held_out","question":ex["prompt"],"answer":a,"type":t})
        if len(items)>=n: break
    return items

HOLDOUT = build_holdout(test_sft_raw, HOLDOUT_N)
EVAL_SET = BENCHMARK + HOLDOUT
print(f"Evaluation set: {len(EVAL_SET)} items ({len(BENCHMARK)} curated + {len(HOLDOUT)} held-out)")

RESULTS = {
 "meta":{"model":"google/gemma-2-2b-it","generated_at":datetime.datetime.now(datetime.timezone.utc).isoformat(),
         "eval_n":len(EVAL_SET),"n_curated":len(BENCHMARK),"n_holdout":len(HOLDOUT),
         "categories":sorted(set(x["category"] for x in EVAL_SET)),"framework":"pytorch-trl-qlora","device":"gpu"},
 "config":{"sft":{"max_steps":SFT_MAX_STEPS,"lr":SFT_LR,"effective_batch":SFT_BATCH*SFT_GRAD_ACC,"lora_r":LORA_R},
           "dpo":{"max_steps":DPO_MAX_STEPS,"lr":DPO_LR,"beta":DPO_BETA}},
 "stages":[], "training":{}}
ALL_RECORDS = {}   # stage_id -> full per-item records (for the teacher/judge pass)

@torch.no_grad()
def _gen(m, prompt, do_sample, temperature=0.7, top_k=40, anti_repeat=False):
    """Generate one completion.
    anti_repeat=True  -> only for the UNTRAINED base model, which otherwise falls
                         into degenerate loops. Adds repetition controls.
    anti_repeat=False -> clean greedy decoding for the trained SFT/DPO models. This
                         is critical: repetition penalties / n-gram bans corrupt the
                         structured <reasoning>...</reasoning><answer>...</answer>
                         output (they block the repeated tag tokens), which tanks
                         both format and answer accuracy. Trained models don't loop,
                         so they need no such crutch."""
    m.eval()
    prev = m.config.use_cache; m.config.use_cache = True
    ids = tokenizer(prompt, return_tensors="pt").to(m.device)
    kw = dict(max_new_tokens=EVAL_MAX_GEN, pad_token_id=tokenizer.pad_token_id, do_sample=do_sample)
    if anti_repeat:
        kw.update(repetition_penalty=1.3, no_repeat_ngram_size=3)
    if do_sample:
        kw.update(temperature=temperature, top_k=top_k)
    out = m.generate(**ids, **kw)
    m.config.use_cache = prev
    txt = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=False)
    for stop in ["<end_of_turn>","<eos>","<pad>"]:
        if stop in txt: txt = txt.split(stop)[0]
    return txt.strip()

def evaluate_model(m, stage_id, stage_name, self_consistency=0, anti_repeat=False, record=True):
    print("\n" + "="*64 + f"\nEVALUATING: {stage_name}\n" + "="*64)
    t0=time.time(); records=[]
    for j,it in enumerate(EVAL_SET):
        prompt=build_prompt(it["question"])
        if self_consistency and self_consistency>1:
            cand=[extract_answer(_gen(m,prompt,True,anti_repeat=anti_repeat)) for _ in range(self_consistency)]
            keys=[_vote_key(a) for a in cand if a is not None]
            if keys:
                win=collections.Counter(keys).most_common(1)[0][0]
                ans=next(a for a in cand if a is not None and _vote_key(a)==win)
            else: ans=None
            # one clean greedy pass so format/steps reflect a real generation
            # (the ANSWER still comes from the majority vote above)
            resp=_gen(m,prompt,False,anti_repeat=anti_repeat)
        else:
            resp=_gen(m,prompt,False,anti_repeat=anti_repeat); ans=extract_answer(resp)
        hr,ha=check_format(resp)
        records.append({"id":it["id"],"category":it["category"],"question":it["question"],
            "expected":it["answer"],"generated":ans,
            "correct":bool(is_correct(it["answer"],ans,it.get("type"))),
            "format_ok":bool(hr and ha),"steps":count_steps(resp),"verify":has_verify(resp),
            "response":resp[:1500]})
        if (j+1)%10==0: print(f"  ...{j+1}/{len(EVAL_SET)}")
    n=len(records); cn=sum(r["correct"] for r in records); fn=sum(r["format_ok"] for r in records)
    overall={"n":n,"correct":cn,"format_correct":fn,
        "answer_accuracy":round(cn/n*100,1),"format_accuracy":round(fn/n*100,1),
        "avg_steps":round(sum(r["steps"] for r in records)/n,2),
        "verify_rate":round(sum(r["verify"] for r in records)/n*100,1)}
    by_cat={}
    for c in sorted(set(r["category"] for r in records)):
        rs=[r for r in records if r["category"]==c]; cc=sum(r["correct"] for r in rs)
        by_cat[c]={"n":len(rs),"correct":cc,"accuracy":round(cc/len(rs)*100,1)}
    stage={"id":stage_id,"name":stage_name,"order":len(RESULTS["stages"]),
        "self_consistency":self_consistency,"overall":overall,"by_category":by_cat,
        "samples":[{k:r[k] for k in ("question","category","expected","generated","correct","format_ok","response")} for r in records[:8]]}
    if record:                        # ablation runs pass record=False to stay
        RESULTS["stages"].append(stage)   # out of the main base->sft->dpo->kto story
        ALL_RECORDS[stage_id] = records   # kept in memory for the teacher/judge pass
    print(f"  Answer accuracy: {overall['answer_accuracy']}%  ({cn}/{n})")
    print(f"  Format accuracy: {overall['format_accuracy']}%  ({fn}/{n})")
    for c,v in by_cat.items(): print(f"     {c:14} {v['accuracy']:5.1f}%  ({v['correct']}/{v['n']})")
    print(f"  (eval time: {(time.time()-t0)/60:.1f} min)")
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return stage

print("Harness ready.")


Evaluation set: 62 items (42 curated + 20 held-out)
Harness ready.


In [6]:
# ============================================================================
# 5. STAGE 0 — BASELINE  (untrained Gemma-2-2B, before any training)
# ============================================================================
print("\n" + "#"*64 + "\n# STAGE 0 / BASELINE\n" + "#"*64)
# anti_repeat=True ONLY here: the untrained model loops without it. The trained
# SFT/DPO stages use clean greedy decoding so their tags aren't corrupted.
evaluate_model(model, "base", "Base Gemma-2-2B (no training)", anti_repeat=True)



################################################################
# STAGE 0 / BASELINE
################################################################

EVALUATING: Base Gemma-2-2B (no training)
  ...10/62
  ...20/62
  ...30/62
  ...40/62
  ...50/62
  ...60/62
  Answer accuracy: 33.9%  (21/62)
  Format accuracy: 0.0%  (0/62)
     arithmetic      11.1%  (1/9)
     held_out        10.0%  (2/20)
     logic_tf        50.0%  (4/8)
     science_tf      62.5%  (5/8)
     short_factual  100.0%  (8/8)
     word_problem    11.1%  (1/9)
  (eval time: 5.0 min)


{'id': 'base',
 'name': 'Base Gemma-2-2B (no training)',
 'order': 0,
 'self_consistency': 0,
 'overall': {'n': 62,
  'correct': 21,
  'format_correct': 0,
  'answer_accuracy': 33.9,
  'format_accuracy': 0.0,
  'avg_steps': 5.69,
  'verify_rate': 0.0},
 'by_category': {'arithmetic': {'n': 9, 'correct': 1, 'accuracy': 11.1},
  'held_out': {'n': 20, 'correct': 2, 'accuracy': 10.0},
  'logic_tf': {'n': 8, 'correct': 4, 'accuracy': 50.0},
  'science_tf': {'n': 8, 'correct': 5, 'accuracy': 62.5},
  'short_factual': {'n': 8, 'correct': 8, 'accuracy': 100.0},
  'word_problem': {'n': 9, 'correct': 1, 'accuracy': 11.1}},
 'samples': [{'question': 'What is the value of (-18)^2 - 2^3?',
   'category': 'arithmetic',
   'expected': '316',
   'generated': '316',
   'correct': True,
   'format_ok': False,
   'response': '```tool_code\n(-18 * -18)\n= 324\n2 ^ 3 = 8\n```\n\n\n```\nAnswer: 316\n```'},
  {'question': 'Compute 7 + 6 * 5 - 4 / 2.',
   'category': 'arithmetic',
   'expected': '35',
   'gene

In [7]:
# ============================================================================
# 6. SFT — supervised fine-tuning with LoRA (QLoRA)
# ============================================================================
_OPTIM = "paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch"
model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", target_modules=LORA_TARGETS, task_type="CAUSAL_LM")

sft_args = SFTConfig(
    output_dir=f"{OUT_DIR}/sft", per_device_train_batch_size=SFT_BATCH,
    gradient_accumulation_steps=SFT_GRAD_ACC, num_train_epochs=1, max_steps=SFT_MAX_STEPS,
    learning_rate=SFT_LR, lr_scheduler_type="cosine", warmup_steps=20,
    logging_steps=10, save_strategy="no", report_to=[],
    max_length=MAX_LENGTH, packing=False, completion_only_loss=True,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=torch.cuda.is_available(), optim=_OPTIM)

print("Starting SFT...  (first steps are slow while it warms up)")
t0 = time.time()
sft_trainer = SFTTrainer(model=model, args=sft_args, train_dataset=sft_train_ds,
                         processing_class=tokenizer, peft_config=lora_config)
sft_trainer.train()
RESULTS["training"]["sft_minutes"] = round((time.time()-t0)/60, 2)
RESULTS["training"]["sft_loss"] = [
    {"step": int(e["step"]), "value": float(e["loss"])}
    for e in sft_trainer.state.log_history if "loss" in e]
model = sft_trainer.model   # PeftModel carrying the trained SFT adapter
# snapshot the SFT adapter so the optional alignment-method ablation (last cell)
# can reload this exact checkpoint and compare DPO/IPO/Hinge/Robust/APO fairly.
try:
    model.save_pretrained(f"{OUT_DIR}/sft_ckpt"); tokenizer.save_pretrained(f"{OUT_DIR}/sft_ckpt")
    print("Saved SFT checkpoint to", f"{OUT_DIR}/sft_ckpt")
except Exception as e:
    print("  (could not save SFT checkpoint:", e, ")")
print(f"SFT done in {RESULTS['training']['sft_minutes']} min.")
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None


Starting SFT...  (first steps are slow while it warms up)


Adding EOS to train dataset:   0%|          | 0/3126 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3126 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3126 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3126 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3126 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss
10,2.878742
20,2.582113
30,2.253719
40,2.074805
50,1.909058
60,1.963032
70,1.943298
80,1.897718
90,1.757428
100,1.887647


Saved SFT checkpoint to /kaggle/working/sft_ckpt
SFT done in 271.44 min.


In [8]:
# ============================================================================
# 7. STAGE 1 — AFTER SFT
# ============================================================================
print("\n" + "#"*64 + "\n# STAGE 1 / AFTER SFT\n" + "#"*64)
evaluate_model(model, "sft", "After SFT")



################################################################
# STAGE 1 / AFTER SFT
################################################################

EVALUATING: After SFT
  ...10/62
  ...20/62
  ...30/62
  ...40/62
  ...50/62
  ...60/62
  Answer accuracy: 72.6%  (45/62)
  Format accuracy: 98.4%  (61/62)
     arithmetic     100.0%  (9/9)
     held_out        50.0%  (10/20)
     logic_tf        62.5%  (5/8)
     science_tf      87.5%  (7/8)
     short_factual  100.0%  (8/8)
     word_problem    66.7%  (6/9)
  (eval time: 8.3 min)


{'id': 'sft',
 'name': 'After SFT',
 'order': 1,
 'self_consistency': 0,
 'overall': {'n': 62,
  'correct': 45,
  'format_correct': 61,
  'answer_accuracy': 72.6,
  'format_accuracy': 98.4,
  'avg_steps': 2.92,
  'verify_rate': 1.6},
 'by_category': {'arithmetic': {'n': 9, 'correct': 9, 'accuracy': 100.0},
  'held_out': {'n': 20, 'correct': 10, 'accuracy': 50.0},
  'logic_tf': {'n': 8, 'correct': 5, 'accuracy': 62.5},
  'science_tf': {'n': 8, 'correct': 7, 'accuracy': 87.5},
  'short_factual': {'n': 8, 'correct': 8, 'accuracy': 100.0},
  'word_problem': {'n': 9, 'correct': 6, 'accuracy': 66.7}},
 'samples': [{'question': 'What is the value of (-18)^2 - 2^3?',
   'category': 'arithmetic',
   'expected': '316',
   'generated': '316',
   'correct': True,
   'format_ok': True,
   'response': '<reasoning>\nCompute (-18)^2 = 324.\nCompute 2^3 = 8.\nCompute 324 − 8 = 316.\n</reasoning>\n<answer>\n316\n</answer>'},
  {'question': 'Compute 7 + 6 * 5 - 4 / 2.',
   'category': 'arithmetic',
   'e

In [9]:
# ============================================================================
# 8. DPO — direct preference optimization on the SFT-adapted model
# ref_model=None -> TRL uses the adapter-disabled base as the frozen reference
# (no second copy of the model, so it fits in GPU memory).
# ============================================================================
model.config.use_cache = False

# free the SFT trainer's optimizer/accelerator state before DPO (the model itself
# stays alive via `model`), then clear the cache — reclaims GPU memory for DPO.
if "sft_trainer" in globals():
    del sft_trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

dpo_args = DPOConfig(
    output_dir=f"{OUT_DIR}/dpo", per_device_train_batch_size=DPO_BATCH,
    gradient_accumulation_steps=DPO_GRAD_ACC, num_train_epochs=1, max_steps=DPO_MAX_STEPS,
    learning_rate=DPO_LR, beta=DPO_BETA, lr_scheduler_type="cosine", warmup_steps=10,
    logging_steps=10, save_strategy="no", report_to=[],
    max_length=MAX_LENGTH,
    precompute_ref_log_probs=True,   # compute reference once up front -> lower peak memory in the loop
    per_device_eval_batch_size=1,    # precompute pass batch (keep small for the T4)
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=torch.cuda.is_available(), optim=_OPTIM)

print("Starting DPO...")
t0 = time.time()
dpo_trainer = DPOTrainer(model=model, ref_model=None, args=dpo_args,
                         train_dataset=dpo_train_ds, processing_class=tokenizer)
dpo_trainer.train()
RESULTS["training"]["dpo_minutes"] = round((time.time()-t0)/60, 2)
RESULTS["training"]["dpo_loss"] = [
    {"step": int(e["step"]), "value": float(e["loss"])}
    for e in dpo_trainer.state.log_history if "loss" in e]
model = dpo_trainer.model
print(f"DPO done in {RESULTS['training']['dpo_minutes']} min.")
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None


Starting DPO...


Adding EOS to train dataset:   0%|          | 0/1559 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1559 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/1559 [00:00<?, ? examples/s]

Computing reference log probs for train dataset:   0%|          | 0/1559 [00:00<?, ?it/s]

Caching reference log probs for train dataset:   0%|          | 0/1559 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/1559 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/1559 [00:00<?, ? examples/s]

Step,Training Loss
10,0.676023
20,0.562846
30,0.475183
40,0.411383
50,0.352678
60,0.381280
70,0.395285
80,0.347000


DPO done in 165.22 min.


In [10]:
# ============================================================================
# 9. STAGE 2 — AFTER SFT + DPO
# (Self-consistency now runs on the FINAL model, after the KTO stage below.)
# ============================================================================
print("\n" + "#"*64 + "\n# STAGE 2 / AFTER SFT + DPO\n" + "#"*64)
evaluate_model(model, "dpo", "After SFT + DPO")



################################################################
# STAGE 2 / AFTER SFT + DPO
################################################################

EVALUATING: After SFT + DPO
  ...10/62
  ...20/62
  ...30/62
  ...40/62
  ...50/62
  ...60/62
  Answer accuracy: 74.2%  (46/62)
  Format accuracy: 96.8%  (60/62)
     arithmetic     100.0%  (9/9)
     held_out        45.0%  (9/20)
     logic_tf        75.0%  (6/8)
     science_tf     100.0%  (8/8)
     short_factual  100.0%  (8/8)
     word_problem    66.7%  (6/9)
  (eval time: 8.4 min)


{'id': 'dpo',
 'name': 'After SFT + DPO',
 'order': 2,
 'self_consistency': 0,
 'overall': {'n': 62,
  'correct': 46,
  'format_correct': 60,
  'answer_accuracy': 74.2,
  'format_accuracy': 96.8,
  'avg_steps': 3.29,
  'verify_rate': 3.2},
 'by_category': {'arithmetic': {'n': 9, 'correct': 9, 'accuracy': 100.0},
  'held_out': {'n': 20, 'correct': 9, 'accuracy': 45.0},
  'logic_tf': {'n': 8, 'correct': 6, 'accuracy': 75.0},
  'science_tf': {'n': 8, 'correct': 8, 'accuracy': 100.0},
  'short_factual': {'n': 8, 'correct': 8, 'accuracy': 100.0},
  'word_problem': {'n': 9, 'correct': 6, 'accuracy': 66.7}},
 'samples': [{'question': 'What is the value of (-18)^2 - 2^3?',
   'category': 'arithmetic',
   'expected': '316',
   'generated': '316',
   'correct': True,
   'format_ok': True,
   'response': '<reasoning>\nCompute (-18)^2 = 324.\nCompute 2^3 = 8.\nCompute 324 − 8 = 316.\n</reasoning>\n<answer>\n316\n</answer>'},
  {'question': 'Compute 7 + 6 * 5 - 4 / 2.',
   'category': 'arithmetic',

In [11]:
# ============================================================================
# 9b. KTO — Kahneman-Tversky Optimization  (Stage 3, on top of SFT + DPO)
# ----------------------------------------------------------------------------
# KTO aligns the model from UNPAIRED, binary feedback (is this completion good
# or bad?) rather than preference pairs. It optimizes a human-aware utility
# (Kahneman-Tversky prospect theory) and often needs no paired data at all.
#
# We derive KTO data straight from the DPO pairs: every "chosen" completion is a
# desirable example (label=True) and every "rejected" one is undesirable
# (label=False). ref_model=None -> the adapter-disabled base is the frozen
# reference (no second model copy, memory-safe on a T4).
#
# The whole stage is wrapped in try/except: if anything goes wrong it is skipped
# and your SFT+DPO model and results (already computed above) are untouched.
# ============================================================================
KTO_DONE = False
try:
    model.config.use_cache = False
    if "dpo_trainer" in globals():
        del dpo_trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ---- build the KTO dataset (prompt, completion, label) from the pairs ----
    kto_rows = []
    for ex in train_dpo_raw:
        p = build_prompt(ex["prompt"])
        kto_rows.append({"prompt": p, "completion": ex["chosen"].strip(),   "label": True})
        kto_rows.append({"prompt": p, "completion": ex["rejected"].strip(), "label": False})
    random.seed(42); random.shuffle(kto_rows)
    kto_train_ds = Dataset.from_list(kto_rows)
    n_good = sum(1 for r in kto_rows if r["label"])
    print(f"KTO dataset: {len(kto_rows)} labeled examples "
          f"({n_good} desirable / {len(kto_rows)-n_good} undesirable)")

    kto_args = KTOConfig(
        output_dir=f"{OUT_DIR}/kto", per_device_train_batch_size=KTO_BATCH,
        gradient_accumulation_steps=KTO_GRAD_ACC, num_train_epochs=1, max_steps=KTO_MAX_STEPS,
        learning_rate=KTO_LR, beta=KTO_BETA,
        desirable_weight=KTO_DESIRABLE_WEIGHT, undesirable_weight=KTO_UNDESIRABLE_WEIGHT,
        lr_scheduler_type="cosine", warmup_steps=10, logging_steps=10,
        save_strategy="no", report_to=[],
        max_length=MAX_LENGTH,
        precompute_ref_log_probs=True, per_device_eval_batch_size=1,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        bf16=torch.cuda.is_available(), optim=_OPTIM)

    print("Starting KTO...")
    t0 = time.time()
    kto_trainer = KTOTrainer(model=model, ref_model=None, args=kto_args,
                             train_dataset=kto_train_ds, processing_class=tokenizer)
    kto_trainer.train()
    RESULTS["training"]["kto_minutes"] = round((time.time()-t0)/60, 2)
    RESULTS["training"]["kto_loss"] = [
        {"step": int(e["step"]), "value": float(e["loss"])}
        for e in kto_trainer.state.log_history if "loss" in e]
    model = kto_trainer.model
    KTO_DONE = True
    print(f"KTO done in {RESULTS['training']['kto_minutes']} min.")
    del kto_trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception as e:
    import traceback; traceback.print_exc()
    print("\nKTO stage skipped:", type(e).__name__, e,
          "\n(Your SFT+DPO model and the results above are unaffected.)")

# ---- STAGE 3 evaluation (only if KTO actually ran) ----
if KTO_DONE:
    print("\n" + "#"*64 + "\n# STAGE 3 / AFTER SFT + DPO + KTO\n" + "#"*64)
    evaluate_model(model, "kto", "After SFT + DPO + KTO")

# ---- STAGE 4 (optional): self-consistency on the FINAL model ----
if USE_SELF_CONSISTENCY:
    tag  = "SFT+DPO+KTO" if KTO_DONE else "SFT+DPO"
    sid  = "kto_sc" if KTO_DONE else "dpo_sc"
    print("\n" + "#"*64 + f"\n# STAGE 4 / SELF-CONSISTENCY (majority vote of {SC_SAMPLES})\n" + "#"*64)
    evaluate_model(model, sid, f"{tag} + Self-Consistency (k={SC_SAMPLES})",
                   self_consistency=SC_SAMPLES)


KTO dataset: 3118 labeled examples (1559 desirable / 1559 undesirable)
Starting KTO...


Adding EOS to train dataset:   0%|          | 0/3118 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3118 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/3118 [00:00<?, ? examples/s]

Extracting KL train dataset:   0%|          | 0/3118 [00:00<?, ? examples/s]

Computing reference log probs for train dataset:   0%|          | 0/1559 [00:00<?, ?it/s]

Caching reference log probs for train dataset:   0%|          | 0/3118 [00:00<?, ? examples/s]

Step,Training Loss
10,0.499152
20,0.490841
30,0.470719
40,0.465822
50,0.468285
60,0.461588


KTO done in 222.49 min.

################################################################
# STAGE 3 / AFTER SFT + DPO + KTO
################################################################

EVALUATING: After SFT + DPO + KTO
  ...10/62
  ...20/62
  ...30/62
  ...40/62
  ...50/62
  ...60/62
  Answer accuracy: 71.0%  (44/62)
  Format accuracy: 96.8%  (60/62)
     arithmetic     100.0%  (9/9)
     held_out        40.0%  (8/20)
     logic_tf        75.0%  (6/8)
     science_tf      87.5%  (7/8)
     short_factual  100.0%  (8/8)
     word_problem    66.7%  (6/9)
  (eval time: 8.3 min)


In [12]:
# ============================================================================
# 9b. TEACHER / JUDGE EVALUATION  (LLM-as-a-judge)
# A stronger model (Gemma-2-9B, 4-bit) grades every stored answer CORRECT /
# INCORRECT — a second, format-agnostic accuracy that doesn't depend on brittle
# string matching. Attach `gemma-2-9b-it` (Transformers) as an Input to enable
# the strong teacher; otherwise it falls back to the trained model as judge.
# Fully wrapped in try/except so it can NEVER break the run.
# ============================================================================
JUDGE_DONE = False
try:
    def _find_judge_dir():
        # prefer a 9B teacher distinct from the student model
        for root, dirs, files in os.walk("/kaggle/input"):
            if "config.json" in files and any(f.endswith(".safetensors") for f in files):
                if "9b" in root.lower():
                    return root
        return None

    judge_dir = _find_judge_dir()
    if judge_dir:
        print("Loading teacher/judge (4-bit) from:", judge_dir)
        _jbnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                   bnb_4bit_compute_dtype=torch.bfloat16,
                                   bnb_4bit_use_double_quant=True)
        judge_tok = AutoTokenizer.from_pretrained(judge_dir)
        if judge_tok.pad_token is None: judge_tok.pad_token = judge_tok.eos_token
        judge_model = AutoModelForCausalLM.from_pretrained(
            judge_dir, quantization_config=_jbnb, device_map={"": 0},
            torch_dtype=torch.bfloat16, attn_implementation="eager")
        judge_model.config.use_cache = True
        JUDGE_NAME = "gemma-2-9b-it (teacher)"
    else:
        print("No 9B teacher attached — using the trained model as judge (weaker). "
              "Attach 'gemma-2-9b-it' (Transformers) as an Input for a stronger teacher.")
        judge_model, judge_tok, JUDGE_NAME = model, tokenizer, "gemma-2-2b (self-judge)"

    @torch.no_grad()
    def judge_one(q, ref, student):
        judge_model.eval()
        prompt = (
            "<start_of_turn>user\n"
            "You are a strict grading assistant. Decide whether the STUDENT ANSWER is correct "
            "for the QUESTION, using the REFERENCE ANSWER as ground truth. Judge only the final "
            "answer's value or meaning — ignore wording, formatting and extra steps. Numerically "
            "equal answers are correct (e.g. 25 and 25/1; -9 and −9). Reply with ONE word: "
            "CORRECT or INCORRECT.\n\n"
            f"QUESTION: {q}\nREFERENCE ANSWER: {ref}\nSTUDENT ANSWER: {student}<end_of_turn>\n"
            "<start_of_turn>model\n")
        ids = judge_tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(judge_model.device)
        out = judge_model.generate(**ids, max_new_tokens=6, do_sample=False,
                                   pad_token_id=judge_tok.pad_token_id)
        txt = judge_tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).lower()
        if "incorrect" in txt: return False
        if "correct" in txt:   return True
        return False

    print("\n" + "#" * 64 + f"\n# TEACHER/JUDGE grading with {JUDGE_NAME}\n" + "#" * 64)
    for stage in RESULTS["stages"]:
        recs = ALL_RECORDS.get(stage["id"], [])
        if not recs:
            continue
        jc = 0; percat = {}
        for r in recs:
            student = r["generated"] if r["generated"] not in (None, "") else (r["response"][-200:])
            v = judge_one(r["question"], r["expected"], student)
            jc += 1 if v else 0
            d = percat.setdefault(r["category"], [0, 0]); d[1] += 1; d[0] += 1 if v else 0
        n = len(recs)
        stage["overall"]["judge_accuracy"] = round(jc / n * 100, 1)
        stage["overall"]["judge_correct"] = jc
        for c, (cc, ct) in percat.items():
            if c in stage["by_category"]:
                stage["by_category"][c]["judge_accuracy"] = round(cc / ct * 100, 1)
        print(f"  {stage['name']:<34} teacher-judged: {stage['overall']['judge_accuracy']:5.1f}%  ({jc}/{n})")

    RESULTS["meta"]["judge_model"] = JUDGE_NAME
    JUDGE_DONE = True
    if judge_dir:   # free the teacher; student stays for the custom-questions cell
        del judge_model, judge_tok
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
except Exception as e:
    import traceback; traceback.print_exc()
    print("\nTeacher/judge step skipped:", type(e).__name__, e,
          "\n(The string-match accuracy already computed above is unaffected.)")


Loading teacher/judge (4-bit) from: /kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/2


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]


################################################################
# TEACHER/JUDGE grading with gemma-2-9b-it (teacher)
################################################################
  Base Gemma-2-2B (no training)      teacher-judged:  33.9%  (21/62)
  After SFT                          teacher-judged:  69.4%  (43/62)
  After SFT + DPO                    teacher-judged:  69.4%  (43/62)
  After SFT + DPO + KTO              teacher-judged:  66.1%  (41/62)


In [13]:
# ============================================================================
# 10. SAVE results.json + adapter + FINAL COMPARISON TABLE
# ============================================================================
RESULTS["training"]["total_minutes"] = round(
    RESULTS["training"].get("sft_minutes",0) + RESULTS["training"].get("dpo_minutes",0)
    + RESULTS["training"].get("kto_minutes",0), 2)

# save the trained LoRA adapter
try:
    model.save_pretrained(f"{OUT_DIR}/final_adapter")
    tokenizer.save_pretrained(f"{OUT_DIR}/final_adapter")
    print("Saved LoRA adapter to", f"{OUT_DIR}/final_adapter")
except Exception as e:
    print("  (could not save adapter:", e, ")")

OUT_PATH = f"{OUT_DIR}/results.json"
with open(OUT_PATH, "w") as f:
    json.dump(RESULTS, f, indent=2, ensure_ascii=False)
print("Saved results to:", OUT_PATH)

print("\n" + "="*72)
print("REASONING IMPROVEMENT — STAGE-BY-STAGE COMPARISON")
print("="*72)
print(f"{'Stage':<34}{'Answer Acc':>12}{'Teacher Acc':>13}{'Format Acc':>12}")
print("-"*72)
for st in RESULTS["stages"]:
    o=st["overall"]
    ja=o.get("judge_accuracy")
    jtxt=f"{ja:>12.1f}%" if isinstance(ja,(int,float)) else f"{'—':>13}"
    print(f"{st['name']:<34}{o['answer_accuracy']:>11.1f}%{jtxt}{o['format_accuracy']:>11.1f}%")
print("-"*72)
if len(RESULTS["stages"])>=2:
    a=RESULTS["stages"][0]["overall"]["answer_accuracy"]; b=RESULTS["stages"][-1]["overall"]["answer_accuracy"]
    print(f"\nReasoning-accuracy gain: {a:.1f}%  ->  {b:.1f}%   (+{b-a:.1f} points)")
print(f"Training time — SFT: {RESULTS['training'].get('sft_minutes','?')} min | "
      f"DPO: {RESULTS['training'].get('dpo_minutes','?')} min | "
      f"KTO: {RESULTS['training'].get('kto_minutes','—')} min | "
      f"Total: {RESULTS['training'].get('total_minutes','?')} min")

print("\n" + "="*72 + "\nresults.json  (copy between the markers if you can't download the file)\n" + "="*72)
print("<<<RESULTS_JSON_START>>>")
print(json.dumps(RESULTS, ensure_ascii=False))
print("<<<RESULTS_JSON_END>>>")


Saved LoRA adapter to /kaggle/working/final_adapter
Saved results to: /kaggle/working/results.json

REASONING IMPROVEMENT — STAGE-BY-STAGE COMPARISON
Stage                               Answer Acc  Teacher Acc  Format Acc
------------------------------------------------------------------------
Base Gemma-2-2B (no training)            33.9%        33.9%        0.0%
After SFT                                72.6%        69.4%       98.4%
After SFT + DPO                          74.2%        69.4%       96.8%
After SFT + DPO + KTO                    71.0%        66.1%       96.8%
------------------------------------------------------------------------

Reasoning-accuracy gain: 33.9%  ->  71.0%   (+37.1 points)
Training time — SFT: 271.44 min | DPO: 165.22 min | KTO: 222.49 min | Total: 659.15 min

results.json  (copy between the markers if you can't download the file)
<<<RESULTS_JSON_START>>>
{"meta": {"model": "google/gemma-2-2b-it", "generated_at": "2026-09-20T15:54:07.576439+00:00", "ev

In [14]:
# ============================================================================
# 11. ALIGNMENT-METHOD ABLATION  (optional)
# ----------------------------------------------------------------------------
# A fair, apples-to-apples comparison of preference-optimization objectives:
# each one starts from the SAME saved SFT checkpoint, trains on your existing
# preference pairs, and is scored on the benchmark. This is a classic ablation
# for a report ("which alignment loss helps most on top of SFT?").
#
# Objectives (all real TRL losses on paired data, selected via DPOConfig.loss_type):
#   sigmoid  -> standard DPO            ipo   -> IPO (Azar et al.)
#   hinge    -> SLiC-style hinge        robust-> robust DPO (label-noise aware)
#   apo_zero -> APO (anchored pref.)
# Choose which to run with ABLATION_METHODS in the config cell. Each is wrapped
# in try/except so one failure never stops the others or the main results.
# (ORPO and SimPO are not in TRL 1.12.0; upgrade TRL if you specifically need them.)
# ============================================================================
if ABLATION_METHODS:
    from peft import PeftModel
    _LOSS_NAMES = {"sigmoid":"DPO (sigmoid)","ipo":"IPO","hinge":"Hinge / SLiC",
                   "robust":"Robust-DPO","apo_zero":"APO-zero"}

    def _load_fresh_sft():
        """Reload base (4-bit) + the saved SFT adapter as a trainable model."""
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        base = AutoModelForCausalLM.from_pretrained(
            model_path, quantization_config=bnb,
            device_map={"": 0} if torch.cuda.is_available() else None,
            torch_dtype=torch.bfloat16, attn_implementation="eager")
        m = PeftModel.from_pretrained(base, f"{OUT_DIR}/sft_ckpt", is_trainable=True)
        m.config.use_cache = False
        return m

    RESULTS["ablation"] = {}
    if not os.path.isdir(f"{OUT_DIR}/sft_ckpt"):
        print("No SFT checkpoint found — skipping ablation.")
    else:
        for method in ABLATION_METHODS:
            try:
                nice = _LOSS_NAMES.get(method, method)
                print("\n" + "#"*64 + f"\n# ABLATION — SFT + {nice}\n" + "#"*64)
                m = _load_fresh_sft()
                abl_args = DPOConfig(
                    output_dir=f"{OUT_DIR}/abl_{method}",
                    per_device_train_batch_size=DPO_BATCH, gradient_accumulation_steps=DPO_GRAD_ACC,
                    num_train_epochs=1, max_steps=ABLATION_MAX_STEPS,
                    learning_rate=DPO_LR, beta=DPO_BETA, loss_type=[method],
                    lr_scheduler_type="cosine", warmup_steps=10, logging_steps=10,
                    save_strategy="no", report_to=[], max_length=MAX_LENGTH,
                    precompute_ref_log_probs=True, per_device_eval_batch_size=1,
                    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
                    bf16=torch.cuda.is_available(), optim=_OPTIM)
                t0 = time.time()
                abl_tr = DPOTrainer(model=m, ref_model=None, args=abl_args,
                                    train_dataset=dpo_train_ds, processing_class=tokenizer)
                abl_tr.train()
                mins = round((time.time()-t0)/60, 2)
                stage = evaluate_model(abl_tr.model, f"abl_{method}", f"SFT + {nice}", record=False)
                RESULTS["ablation"][method] = {
                    "name": nice, "minutes": mins,
                    "answer_accuracy": stage["overall"]["answer_accuracy"],
                    "format_accuracy": stage["overall"]["format_accuracy"]}
                del abl_tr, m
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
            except Exception as e:
                import traceback; traceback.print_exc()
                print(f"  ablation '{method}' skipped:", type(e).__name__, e)

        # re-save results.json (now including the ablation) + print a comparison
        try:
            with open(f"{OUT_DIR}/results.json", "w") as f:
                json.dump(RESULTS, f, indent=2, ensure_ascii=False)
        except Exception as e:
            print("  (could not re-save results.json:", e, ")")
        if RESULTS["ablation"]:
            print("\n" + "="*54)
            print("ALIGNMENT-METHOD ABLATION  (all from the same SFT model)")
            print("="*54)
            print(f"{'Method':<18}{'Answer':>10}{'Format':>10}{'Min':>8}")
            print("-"*54)
            for meth, v in RESULTS["ablation"].items():
                print(f"{v['name']:<18}{v['answer_accuracy']:>9.1f}%{v['format_accuracy']:>9.1f}%{v['minutes']:>8.1f}")
            print("-"*54)
else:
    print("Ablation disabled (ABLATION_METHODS is empty).")


Ablation disabled (ABLATION_METHODS is empty).


In [15]:
# ============================================================================
# 11. JUDGE / CUSTOM QUESTIONS  (try the final model on your own prompts)
# ============================================================================
judge_questions = [
    "What is the value of (-18)^2 - 2^3?",
    "A shop offers 30% off, then an extra 10% off the discounted price. What single percentage discount is that? Give a percent.",
    "All squares are rectangles. Some rectangles are red. Does it follow that some squares are red? Answer TRUE or FALSE.",
]
for i, q in enumerate(judge_questions, 1):
    resp = _gen(model, build_prompt(q), do_sample=False)
    print("\n" + "="*64 + f"\n[Q{i}] {q}\n" + "-"*64)
    print(resp)
    print("-"*64 + f"\nExtracted answer: {extract_answer(resp)}")
print("\nDone. Edit `judge_questions` to try your own prompts.")



[Q1] What is the value of (-18)^2 - 2^3?
----------------------------------------------------------------
<reasoning>
Compute (-18)^2 = 324.
Compute 2^3 = 8.
Compute 324 − 8 = 316.
</reasoning>
<answer>
316
</answer>
----------------------------------------------------------------
Extracted answer: 316

[Q2] A shop offers 30% off, then an extra 10% off the discounted price. What single percentage discount is that? Give a percent.
----------------------------------------------------------------
<reasoning>
Convert 30% to 30 ÷ 100.
Compute discount_1 = (30 ÷ 100) × original_price.
Compute discounted_price = original_price − discount_1.
Convert 10% to 10 ÷ 100.
Compute discount_2 = (10 ÷ 100) × discounted_price.
Compute final_price = discounted_price − discount_2.
Compute fraction = final_price ÷ original_price.
Compute percentage = fraction × 100.
</reasoning>
<answer>
35.0000%
</answer>
----------------------------------------------------------------
Extracted answer: 35.0000%

[Q3] Al